# Análise de Drift — Mês 1

Este notebook fecha o **Mês 1** do cronograma de pesquisa: ele consome arquivos
prontos do disco (gerados por `drift_analise/plot_covariate_shift.py` e
`model_analise/compute_trajectory_errors.py`), enriquece o `dataset.csv` com contexto por
jogo, expande para granularidade de **janela** dentro de jogo, traz uma amostra
de erros **por trajetória** e produz as figuras descritivas correspondentes em
`Relas/results/mes1/` (versões PT e EN).

> **Importante** — toda a lógica de cálculo pesado (agregações sobre os
> pickles, predição dos modelos) vive nos scripts. O notebook só lê arquivos
> resultantes e desenha. Rode `python drift_analise/plot_covariate_shift.py` e
> `python model_analise/compute_trajectory_errors.py` antes de "Run All".


In [ ]:
# Imports — sem duplicação
import os
import sys
import math
from pathlib import Path

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
from matplotlib.ticker import PercentFormatter

from scipy import stats


In [ ]:
# =========================
# Estilo global dos plots (idêntico ao notebook anterior)
# =========================
PLOT_STYLE = {
    "font.family": "serif",
    "font.size": 13,
    "axes.titlesize": 14,
    "axes.labelsize": 13,
    "legend.fontsize": 11,
    "xtick.labelsize": 11,
    "ytick.labelsize": 11,
    "mathtext.fontset": "cm",
    "lines.linewidth": 2,
    "axes.grid": True,
    "grid.alpha": 0.3,
    "figure.autolayout": True,
}
plt.rcParams.update(PLOT_STYLE)

def T(pt, en, lang):
    """Helper para selecionar string por idioma nos plots PT/EN."""
    return en if lang == "en" else pt


In [ ]:
# Caminhos / constantes (estrutura reorganizada)
# A raiz do projeto contém `requirements.txt`. Walk up até encontrar.
ROOT = Path(".").resolve()
while ROOT != ROOT.parent and not (ROOT / "requirements.txt").exists():
    ROOT = ROOT.parent

DATASET_DIR = ROOT / "drift_analise" / "dataset"
COV_DIR     = ROOT / "covariate_shift_out"
MODEL_DIR   = ROOT / "model"
OUT_DIR     = ROOT / "Relas" / "results" / "mes1"
OUT_DIR.mkdir(parents=True, exist_ok=True)

LANGS = ["pt", "en"]
EXCLUDE_YEARS = {2020, 2021}   # mantém critério do paper
N_BOOT = 2000
SEED = 42
CI = 95
BASELINE_YEAR = 2019

print("ROOT       :", ROOT)
print("DATASET_DIR:", DATASET_DIR)
print("OUT_DIR    :", OUT_DIR)


In [ ]:
# Helpers reutilizados em todo o notebook

def save_fig(fig, name):
    """Salva 1 figura em PT e EN (PDF) na OUT_DIR.
    `name` deve ser sem extensão; sufixos `_pt.pdf` e `_en.pdf` são adicionados.
    """
    for lang in LANGS:
        out = OUT_DIR / f"{name}_{lang}.pdf"
        fig.savefig(out, format="pdf", bbox_inches="tight")
    plt.close(fig)


def bootstrap_ci_mean(values, n_boot=N_BOOT, ci=CI, seed=SEED):
    vals = np.asarray(values, dtype=float)
    vals = vals[np.isfinite(vals)]
    if vals.size == 0:
        return (np.nan, np.nan)
    rng = np.random.default_rng(seed)
    n = vals.size
    boot = np.empty(n_boot)
    for b in range(n_boot):
        boot[b] = np.mean(vals[rng.integers(0, n, size=n)])
    lo = np.percentile(boot, (100 - ci) / 2)
    hi = np.percentile(boot, 100 - (100 - ci) / 2)
    return float(lo), float(hi)


def bootstrap_linreg_ci(x, y, x_grid=None, n_boot=N_BOOT, ci=CI, seed=SEED):
    """Retorna dict com slope, intercept, mean_pred, ci_lo, ci_hi sobre x_grid."""
    x = np.asarray(x, dtype=float); y = np.asarray(y, dtype=float)
    m = np.isfinite(x) & np.isfinite(y)
    x, y = x[m], y[m]
    if x.size < 3:
        return None
    if x_grid is None:
        x_grid = np.linspace(np.min(x), np.max(x), 120)
    rng = np.random.default_rng(seed)
    n = x.size
    preds = np.empty((n_boot, x_grid.size))
    for b in range(n_boot):
        idx = rng.integers(0, n, size=n)
        sx, sy = x[idx], y[idx]
        if np.std(sx) < 1e-12:
            preds[b] = np.nan
            continue
        slope, intercept, *_ = stats.linregress(sx, sy)
        preds[b] = slope * x_grid + intercept
    mean_pred = np.nanmean(preds, axis=0)
    lo = np.nanpercentile(preds, (100 - ci) / 2, axis=0)
    hi = np.nanpercentile(preds, 100 - (100 - ci) / 2, axis=0)
    return {"x_grid": x_grid, "mean": mean_pred, "lo": lo, "hi": hi}


## Pré-checagem dos arquivos de entrada

In [ ]:
# Verifica os arquivos esperados; aviso (não erro fatal) se faltar.
EXPECTED = {
    "drift_analise/dataset/dataset.csv":            DATASET_DIR / "dataset.csv",
    "covariate_shift_out/ks_wd_per_dataset.csv":    COV_DIR / "ks_wd_per_dataset.csv",
    "covariate_shift_out/per_game_context.csv":     COV_DIR / "per_game_context.csv",
    "covariate_shift_out/per_window_drift.csv":     COV_DIR / "per_window_drift.csv",
    "covariate_shift_out/trajectory_errors_sample": (
        COV_DIR / "trajectory_errors_sample.parquet",
        COV_DIR / "trajectory_errors_sample.csv",
    ),
}

print("=" * 64)
print("Pré-checagem")
print("=" * 64)
missing = []
for label, p in EXPECTED.items():
    if isinstance(p, tuple):
        ok = any(pp.exists() for pp in p)
        shown = next((str(pp) for pp in p if pp.exists()), str(p[0]))
    else:
        ok = p.exists()
        shown = str(p)
    flag = "OK " if ok else "FAL"
    print(f"  [{flag}] {label:50s} -> {shown}")
    if not ok:
        missing.append(label)

if missing:
    print("\n[!] Arquivos ausentes:")
    for m_ in missing:
        print(f"    - {m_}")
    print("\nRode os scripts antes do notebook:")
    print("    python drift_analise/plot_covariate_shift.py        # gera per_game_context.csv e per_window_drift.csv")
    print("    python model_analise/compute_trajectory_errors.py   # gera trajectory_errors_sample.parquet")
else:
    print("\n[ok] Tudo pronto para análise.")


# 1. Dataset enriquecido por jogo

A partir de `dataset.csv` (1 linha por (jogo, horizonte, métrica, modelo)) e
`per_game_context.csv` (1 linha por jogo com contexto cinemático: percentis de
velocidade/aceleração/curvatura, nº de robôs, duração estimada), montamos
`dataset_enriched.csv`. Esse é o DataFrame-mestre usado nas seções seguintes.


In [ ]:
df = pd.read_csv(DATASET_DIR / "dataset.csv")
df_ctx = pd.read_csv(COV_DIR / "per_game_context.csv")

df["year"]  = pd.to_numeric(df["year"], errors="coerce")
df["value"] = pd.to_numeric(df["value"], errors="coerce")

print("dataset.csv         shape:", df.shape, "  jogos únicos:", df["log_file"].nunique())
print("per_game_context.csv shape:", df_ctx.shape, "  jogos únicos:", df_ctx["log_file"].nunique())
df_ctx.head(3)


In [ ]:
# JOIN por log_file -- mantém todas as linhas do dataset.csv (left join)
ctx_cols = [c for c in df_ctx.columns if c != "log_file"]
df_all = df.merge(df_ctx, on="log_file", how="left", suffixes=("", "_ctx"))

# salva a versão enriquecida (na raiz do projeto, não na pasta de figuras)
df_all.to_csv(DATASET_DIR / "dataset_enriched.csv", index=False)
print("dataset_enriched.csv salvo em:", DATASET_DIR / "dataset_enriched.csv")

# DataFrame filtrado para análises principais (excluir 2020 e 2021)
df_all_filtered = df_all[~df_all["year"].isin(EXCLUDE_YEARS)].copy()
print("Anos no df_all:         ", sorted(df_all["year"].dropna().unique().tolist()))
print("Anos no df_all_filtered:", sorted(df_all_filtered["year"].dropna().unique().tolist()))


In [ ]:
# Inspeção rápida das colunas novas vindas do contexto
new_cols = [c for c in df_ctx.columns if c not in {"log_file"}]
print("Colunas novas no enriched:", new_cols)
df_all.head(3)


# 2. Drift em granularidade de janela

`per_window_drift.csv` traz KS e Wasserstein **por janela** de
`N_TRAJS_PER_WINDOW=500` trajetórias contra o baseline (treino, jogos
2019/proc_set_1+2). Aqui mostramos que existe variação **intra-ano** e
**intra-jogo** que a granularidade anual original mascarava.


In [ ]:
df_win = pd.read_csv(COV_DIR / "per_window_drift.csv")
df_win["year"] = pd.to_numeric(df_win["year"], errors="coerce")

print("per_window_drift.csv  shape:", df_win.shape)
print("Janelas por ano:")
print(df_win.groupby("year")["window_id"].count().rename("n_janelas"))

# ordem temporal global das janelas (ano crescente, depois match_id, depois window_id)
df_win = df_win.sort_values(["year", "match_id", "window_id"]).reset_index(drop=True)
df_win["global_idx"] = np.arange(len(df_win))
df_win.head()


In [ ]:
# Mapa de cores estável por ano (usado nas seções 2 e 4)
years_all = sorted(df_win["year"].dropna().unique().astype(int).tolist())
cmap = plt.get_cmap("viridis")
YEAR_COLORS = {int(y): cmap(i / max(1, len(years_all) - 1)) for i, y in enumerate(years_all)}


def make_plot_window_timeline(metric, lang):
    fig, ax = plt.subplots(figsize=(11, 5))
    for y in years_all:
        sub = df_win[df_win["year"] == y]
        ax.scatter(sub["global_idx"], sub[metric],
                   color=YEAR_COLORS[y], s=18, alpha=0.7, label=str(y))
    ax.set_xlabel(T("Janela (ordem temporal global)", "Window (global temporal order)", lang))
    ax.set_ylabel(metric.replace("_window", "").upper().replace("_", " "))
    ax.set_title(T(f"Drift por janela — {metric}", f"Per-window drift — {metric}", lang))
    ax.legend(title=T("Ano", "Year", lang), ncol=2, fontsize=9, loc="best")
    return fig


for metric in ["ks_speed_window", "wd_speed_window", "ks_accel_window", "wd_accel_window"]:
    for lang in LANGS:
        fig = make_plot_window_timeline(metric, lang)
        save_fig(fig, f"2a_window_timeline_{metric}")
print("-> 2a_window_timeline_*.pdf")


In [ ]:
def make_plot_window_box(metric, lang):
    fig, ax = plt.subplots(figsize=(9, 5))
    data = [df_win.loc[df_win["year"] == y, metric].dropna().values for y in years_all]
    bp = ax.boxplot(data, labels=[str(y) for y in years_all],
                    patch_artist=True, showfliers=True)
    for patch, y in zip(bp["boxes"], years_all):
        patch.set_facecolor(YEAR_COLORS[y]); patch.set_alpha(0.55)
    ax.set_xlabel(T("Ano", "Year", lang))
    ax.set_ylabel(metric.replace("_window", "").upper().replace("_", " "))
    ax.set_title(T(f"Distribuição intra-ano de {metric}",
                    f"Intra-year distribution of {metric}", lang))
    return fig


for metric in ["ks_speed_window", "wd_speed_window", "ks_accel_window", "wd_accel_window"]:
    for lang in LANGS:
        fig = make_plot_window_box(metric, lang)
        save_fig(fig, f"2b_window_box_{metric}")
print("-> 2b_window_box_*.pdf")


In [ ]:
# Tabela: variabilidade intra-ano
agg = (df_win
       .groupby("year")[["ks_speed_window", "wd_speed_window",
                          "ks_accel_window", "wd_accel_window"]]
       .agg([("p50", "median"), ("p90", lambda s: np.nanpercentile(s, 90))])
      )
agg.columns = ["__".join(c) for c in agg.columns]
agg = agg.reset_index()
agg.to_csv(OUT_DIR / "window_intra_year_var.csv", index=False)
print("-> window_intra_year_var.csv")
agg


# 3. Erros por trajetória — amostra

Carrega `trajectory_errors_sample.parquet` (gerado por
`model_analise/compute_trajectory_errors.py`). Permite analisar a **cauda** dos erros, não
só a média por jogo.


In [ ]:
parquet_path = COV_DIR / "trajectory_errors_sample.parquet"
csv_path     = COV_DIR / "trajectory_errors_sample.csv"
if parquet_path.exists():
    df_traj = pd.read_parquet(parquet_path)
elif csv_path.exists():
    df_traj = pd.read_csv(csv_path)
else:
    raise FileNotFoundError(
        "Nem trajectory_errors_sample.parquet nem .csv foram encontrados. "
        "Rode `python model_analise/compute_trajectory_errors.py` antes."
    )

df_traj["year"] = pd.to_numeric(df_traj["year"], errors="coerce")
print("Linhas:", len(df_traj))
print(df_traj.groupby(["year", "model", "horizon"])["traj_id"].count()
      .rename("n_trajs"))
df_traj.head()


In [ ]:
# Histograma com escala log no eixo Y, comparando 2019 vs último ano disponível,
# por (modelo, horizonte).
years_in_traj = sorted(df_traj["year"].dropna().unique().astype(int).tolist())
year_last = max([y for y in years_in_traj if y not in EXCLUDE_YEARS],
                default=max(years_in_traj))


def make_plot_ade_traj_hist(model, horizon, lang):
    fig, ax = plt.subplots(figsize=(8, 5))
    base = df_traj[(df_traj["model"] == model) &
                   (df_traj["horizon"] == horizon) &
                   (df_traj["year"] == BASELINE_YEAR)]["ade_traj"].dropna()
    last = df_traj[(df_traj["model"] == model) &
                   (df_traj["horizon"] == horizon) &
                   (df_traj["year"] == year_last)]["ade_traj"].dropna()
    if base.empty and last.empty:
        ax.text(0.5, 0.5, T("Sem dados", "No data", lang), ha="center", va="center")
        return fig

    pool = pd.concat([base, last])
    if pool.empty:
        ax.text(0.5, 0.5, T("Sem dados", "No data", lang), ha="center", va="center")
        return fig
    lo = np.nanpercentile(pool, 0.5)
    hi = np.nanpercentile(pool, 99.5)
    bins = np.linspace(max(0, lo), hi, 60) if hi > lo else 30

    if not base.empty:
        ax.hist(base, bins=bins, alpha=0.55, label=str(BASELINE_YEAR),
                color=YEAR_COLORS.get(BASELINE_YEAR, "C0"), edgecolor="white")
    if not last.empty:
        ax.hist(last, bins=bins, alpha=0.55, label=str(year_last),
                color=YEAR_COLORS.get(year_last, "C3"), edgecolor="white")
    ax.set_yscale("log")
    ax.set_xlabel(T("ADE por trajetória (mm)", "Per-trajectory ADE (mm)", lang))
    ax.set_ylabel(T("Frequência (escala log)", "Frequency (log scale)", lang))
    ax.set_title(T(f"Distribuição de ADE — {model} {horizon}",
                    f"ADE distribution — {model} {horizon}", lang))
    ax.legend()
    return fig


for model in ["Seq2Seq", "Kalman"]:
    for horizon in ["30→15", "60→30"]:
        for lang in LANGS:
            fig = make_plot_ade_traj_hist(model, horizon, lang)
            save_fig(fig, f"3a_ade_traj_hist_{model}_{horizon.replace('→','_')}")
print("-> 3a_ade_traj_hist_*.pdf")


In [ ]:
# Tabela tail_stats: p50, p90, p99 de ade_traj por (year, model, horizon)
def _q(p):
    def _f(x): return float(np.nanpercentile(x, p))
    _f.__name__ = f"p{p}"
    return _f

tail_stats = (df_traj
              .groupby(["year", "model", "horizon"])["ade_traj"]
              .agg([_q(50), _q(90), _q(99), "count"])
              .reset_index()
              .rename(columns={"count": "n_trajs"})
             )
tail_stats.to_csv(OUT_DIR / "tail_stats.csv", index=False)
print("-> tail_stats.csv")
tail_stats


In [ ]:
def make_plot_p99_p50_ratio(lang):
    ratio = (df_traj
             .groupby(["year", "model", "horizon"])["ade_traj"]
             .agg([("p50", lambda s: np.nanpercentile(s, 50)),
                   ("p99", lambda s: np.nanpercentile(s, 99))])
             .reset_index())
    ratio["p99_p50"] = ratio["p99"] / ratio["p50"]

    fig, ax = plt.subplots(figsize=(9, 5))
    for (model, horizon), sub in ratio.groupby(["model", "horizon"]):
        sub = sub.sort_values("year")
        ax.plot(sub["year"], sub["p99_p50"], marker="o",
                label=f"{model} {horizon}")
    ax.set_xlabel(T("Ano", "Year", lang))
    ax.set_ylabel(T("Razão p99 / p50 (ADE)", "p99 / p50 ratio (ADE)", lang))
    ax.set_title(T("Cauda relativa do erro ao longo dos anos",
                    "Relative error tail across years", lang))
    ax.legend()
    return fig


for lang in LANGS:
    fig = make_plot_p99_p50_ratio(lang)
    save_fig(fig, "3c_p99_p50_ratio")
print("-> 3c_p99_p50_ratio_*.pdf")


# 4. Descritivas na granularidade por jogo

Agora cada **ponto** vira um jogo (não um ano). Cruza ADE com features
cinemáticas do jogo para investigar a hipótese de que jogos de cauda
extrema têm pior previsão.


In [ ]:
# DataFrame "1 linha por (log_file, year, model, horizon)" só com ADE,
# já com colunas enriquecidas do contexto do jogo.
df_ade_game = (df_all_filtered[
        (df_all_filtered["metric"] == "ADE")
        & (df_all_filtered["model"].isin(["Seq2Seq", "Kalman"]))
    ]
    .groupby(["log_file", "year", "model", "horizon"], as_index=False)
    .agg({
        "value": "mean",
        "ks_speed": "first", "ks_accel": "first", "ks_turn": "first",
        "wd_speed": "first", "wd_accel": "first", "wd_turn": "first",
        "speed_p90": "first", "accel_p90": "first",
        "speed_p99": "first", "accel_p99": "first",
        "match_id":  "first",
    })
    .rename(columns={"value": "ade"})
)
print("Linhas df_ade_game:", len(df_ade_game))
df_ade_game.head()


In [ ]:
def make_plot_ade_year_per_game(lang):
    fig, ax = plt.subplots(figsize=(9, 5))
    sub_30 = df_ade_game[df_ade_game["horizon"] == "30→15"].copy()

    rng = np.random.default_rng(SEED)
    sub_30["x_jit"] = sub_30["year"] + rng.uniform(-0.18, 0.18, size=len(sub_30))

    model_colors = {"Seq2Seq": "#2A6FDB", "Kalman": "#D1495B"}
    for model, df_m in sub_30.groupby("model"):
        ax.scatter(df_m["x_jit"], df_m["ade"], color=model_colors[model],
                   alpha=0.45, s=28, label=T(f"{model} (jogo)", f"{model} (game)", lang))

        # média anual com IC bootstrap
        for y, df_y in df_m.groupby("year"):
            mu = df_y["ade"].mean()
            lo, hi = bootstrap_ci_mean(df_y["ade"].values)
            ax.errorbar(y, mu, yerr=[[mu - lo], [hi - mu]], fmt="o",
                        color=model_colors[model], ecolor=model_colors[model],
                        markersize=8, markeredgecolor="white", elinewidth=2,
                        capsize=4)

    ax.set_xlabel(T("Ano", "Year", lang))
    ax.set_ylabel(T("ADE — horizonte 30→15 (mm)", "ADE — 30→15 horizon (mm)", lang))
    ax.set_title(T("ADE por jogo — Seq2Seq vs Kalman (IC95% bootstrap nas médias)",
                    "Per-game ADE — Seq2Seq vs Kalman (95% bootstrap CI on means)", lang))
    ax.legend()
    return fig


for lang in LANGS:
    fig = make_plot_ade_year_per_game(lang)
    save_fig(fig, "4a_ade_year_per_game")
print("-> 4a_ade_year_per_game_*.pdf")


In [ ]:
def make_plot_ade_vs_feature(feature, lang):
    sub = df_ade_game[(df_ade_game["model"] == "Seq2Seq")
                      & (df_ade_game["horizon"] == "30→15")
                      & df_ade_game[feature].notna()
                      & df_ade_game["ade"].notna()]
    if len(sub) < 3:
        fig, ax = plt.subplots(figsize=(7, 5))
        ax.text(0.5, 0.5, T("Dados insuficientes", "Insufficient data", lang),
                ha="center", va="center")
        return fig

    fig, ax = plt.subplots(figsize=(8, 5))
    for y, dfy in sub.groupby("year"):
        ax.scatter(dfy[feature], dfy["ade"], color=YEAR_COLORS.get(int(y), "C0"),
                   s=42, alpha=0.85, label=str(int(y)),
                   edgecolor="white", linewidth=0.6)

    fit = bootstrap_linreg_ci(sub[feature].values, sub["ade"].values)
    if fit is not None:
        ax.plot(fit["x_grid"], fit["mean"], color="black", lw=2,
                label=T("Regressão (média)", "Regression (mean)", lang))
        ax.fill_between(fit["x_grid"], fit["lo"], fit["hi"],
                        color="black", alpha=0.12,
                        label=T("IC95% bootstrap", "95% bootstrap CI", lang))

    pretty = {
        "speed_p90": T("p90 da velocidade do jogo (mm/s)",
                        "Game speed p90 (mm/s)", lang),
        "accel_p90": T("p90 da aceleração do jogo (mm/s²)",
                        "Game accel p90 (mm/s²)", lang),
        "ks_speed":  T("KS de velocidade vs baseline",
                        "Speed KS vs baseline", lang),
    }.get(feature, feature)

    ax.set_xlabel(pretty)
    ax.set_ylabel(T("ADE Seq2Seq 30→15 (mm)", "ADE Seq2Seq 30→15 (mm)", lang))
    ax.set_title(T(f"ADE vs {pretty}", f"ADE vs {pretty}", lang))
    ax.legend(ncol=2, fontsize=9)
    return fig


for feature in ["speed_p90", "accel_p90", "ks_speed"]:
    for lang in LANGS:
        fig = make_plot_ade_vs_feature(feature, lang)
        save_fig(fig, f"4b_ade_vs_{feature}")
print("-> 4b_ade_vs_*.pdf")


In [ ]:
def make_plot_corr_heatmap(lang):
    feats = ["ks_speed", "ks_accel", "ks_turn",
             "wd_speed", "wd_accel", "wd_turn",
             "speed_p90", "accel_p90"]
    # reconstrói visão "1 linha por jogo" com ADE de cada modelo
    pivot = (df_ade_game[df_ade_game["horizon"] == "30→15"]
             .pivot_table(index=["log_file", "year"], columns="model",
                          values="ade", aggfunc="mean")
             .reset_index()
             .rename(columns={"Seq2Seq": "ade_seq2seq", "Kalman": "ade_kalman"}))
    ctx = (df_ade_game[df_ade_game["horizon"] == "30→15"]
           .groupby("log_file", as_index=False)[feats].first())
    M = pivot.merge(ctx, on="log_file", how="left")
    M = M[~M["year"].isin(EXCLUDE_YEARS)]
    cols = ["ade_seq2seq", "ade_kalman"] + feats
    corr = M[cols].corr(method="spearman")

    fig, ax = plt.subplots(figsize=(8, 7))
    im = ax.imshow(corr.values, vmin=-1, vmax=1, cmap="RdBu_r")
    ax.set_xticks(range(len(cols))); ax.set_yticks(range(len(cols)))
    ax.set_xticklabels(cols, rotation=45, ha="right")
    ax.set_yticklabels(cols)
    for i in range(len(cols)):
        for j in range(len(cols)):
            v = corr.values[i, j]
            ax.text(j, i, f"{v:.2f}", ha="center", va="center",
                    color="white" if abs(v) > 0.55 else "black", fontsize=9)
    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    ax.set_title(T("Correlação Spearman (jogos, anos ≠ 2020/2021)",
                    "Spearman correlation (games, years ≠ 2020/2021)", lang))
    return fig


for lang in LANGS:
    fig = make_plot_corr_heatmap(lang)
    save_fig(fig, "4d_corr_spearman")
print("-> 4d_corr_spearman_*.pdf")


# 5. Sanidade & índice de outputs

Tabela resumo da nova granularidade + verificações soft + listagem dos
artefatos gerados (separando "infra para Mês 2" de "outputs descritivos").


In [ ]:
# Tabela resumo: por ano -> n_jogos, n_trajetorias_total, n_janelas_total
ctx_g = df_ctx.copy()
ctx_g["year"] = pd.to_numeric(ctx_g["year"], errors="coerce")

summary = (ctx_g.groupby("year", as_index=False)
           .agg(n_jogos=("proc_set_file", "nunique"),
                n_trajetorias_total=("n_trajectories", "sum"))
          )

win_per_year = (df_win.groupby("year", as_index=False)
                .agg(n_janelas_total=("window_id", "count")))

summary = summary.merge(win_per_year, on="year", how="left").fillna({"n_janelas_total": 0})
summary["n_janelas_total"] = summary["n_janelas_total"].astype(int)
summary.to_csv(OUT_DIR / "summary_granularity.csv", index=False)
print("-> summary_granularity.csv")
summary


In [ ]:
# Verificações soft (warnings via print, não levantam exceção)
print("=" * 64)
print("Sanidade")
print("=" * 64)

low_traj = df_ctx[df_ctx["n_trajectories"] < 10]
if len(low_traj):
    print(f"[!] {len(low_traj)} jogo(s) com <10 trajetórias:")
    print(low_traj[["proc_set_file", "year", "n_trajectories"]].to_string(index=False))
else:
    print("[ok] todos os jogos têm >= 10 trajetórias.")

few_games = summary[summary["n_jogos"] < 2]
if len(few_games):
    print(f"[!] {len(few_games)} ano(s) com <2 jogos:")
    print(few_games[["year", "n_jogos"]].to_string(index=False))
else:
    print("[ok] todos os anos têm >= 2 jogos.")

ks_wd_cols = ["ks_accel", "ks_speed", "ks_turn", "wd_accel", "wd_speed", "wd_turn"]
nan_count = df_all[ks_wd_cols].isna().sum()
if nan_count.sum() > 0:
    print("[!] NaN em colunas KS/WD do dataset_enriched:")
    print(nan_count[nan_count > 0])
else:
    print("[ok] sem NaN em colunas KS/WD do dataset_enriched.")


## Arquivos gerados

**Infra para Mês 2** (granularidade fina, alimenta detectores online):

- `dataset_enriched.csv` — `dataset.csv` + contexto cinemático por jogo
- `covariate_shift_out/per_game_context.csv` — 1 linha por jogo
- `covariate_shift_out/per_window_drift.csv` — KS/WD por janela (N=500 trajetórias)
- `covariate_shift_out/trajectory_errors_sample.parquet` — ADE/FDE por trajetória (amostra)

**Outputs descritivos** (figuras e tabelas em `Relas/results/mes1/`):

- `2a_window_timeline_*.pdf` — timeline de KS/WD por janela
- `2b_window_box_*.pdf` — boxplots intra-ano de KS/WD
- `window_intra_year_var.csv` — mediana e p90 intra-ano
- `3a_ade_traj_hist_*.pdf` — histograma log-y de ADE por trajetória
- `tail_stats.csv` — p50/p90/p99 de ADE por (ano, modelo, horizonte)
- `3c_p99_p50_ratio_*.pdf` — razão p99/p50 ao longo dos anos
- `4a_ade_year_per_game_*.pdf` — ADE por jogo (scatter+jitter) com IC
- `4b_ade_vs_{feature}_*.pdf` — ADE vs speed_p90 / accel_p90 / ks_speed
- `4d_corr_spearman_*.pdf` — heatmap de correlação
- `summary_granularity.csv` — n_jogos / n_trajetórias / n_janelas por ano

Cada PDF é gerado em duas versões (`_pt.pdf` e `_en.pdf`) na pasta
`Relas/results/mes1/`.


# Mês 2 — Detecção formal de drift

A partir desta seção paramos de inferir drift apenas pela visualização e
passamos a usar **detectores formais** da literatura. Cada detector emite
"alarmes" — pontos do stream onde a estatística de teste indica mudança
significativa. No final do notebook (Seção 9) os alarmes são reunidos numa
tabela única, comparados com pontos de quebra retrospectivos do `ruptures`
e discutidos à luz de eventos conhecidos da liga.

> **Dependências novas**: `river` (detectores online) e `ruptures` (offline).
> Instale com `pip install river ruptures` se ainda não tiver.


In [ ]:
# =========================
# Imports e setup do Mês 2
# =========================
import river
from river import drift as river_drift
import ruptures as rpt

print("river   ", river.__version__)
print("ruptures", rpt.__version__)

# Pasta de saída específica do Mês 2 (não mistura com Mês 1)
OUT_DIR_MES2 = ROOT / "Relas" / "results" / "mes2"
OUT_DIR_MES2.mkdir(parents=True, exist_ok=True)


def save_fig_mes2(fig, name):
    """Salva 1 figura em PT e EN (PDF) na OUT_DIR_MES2."""
    for lang in LANGS:
        out = OUT_DIR_MES2 / f"{name}_{lang}.pdf"
        fig.savefig(out, format="pdf", bbox_inches="tight")
    plt.close(fig)


# Dicionário global onde cada detector deposita seus alarmes
# Estrutura: ALARMS[detector_name] = list[int] de índices no stream correspondente
ALARMS = {}
print("OUT_DIR_MES2:", OUT_DIR_MES2)


# 6. Concept drift no erro do modelo — ADWIN e Page-Hinkley

Aplicamos os dois detectores sobre a série temporal de `ade_traj` do
**Seq2Seq, horizonte 30→15**. Como `df_traj` é uma amostra (3 jogos por ano,
seed fixo), tratamos como stream contínuo respeitando a ordem
**(year, match_id, traj_id)**. Cada `update()` representa a chegada de uma
nova trajetória.


In [ ]:
# Stream temporal do erro Seq2Seq 30→15 (ordem global)
df_stream = (df_traj[
        (df_traj["model"] == "Seq2Seq")
        & (df_traj["horizon"] == "30→15")
    ]
    .sort_values(["year", "match_id", "traj_id"])
    .reset_index(drop=True)
)
df_stream["global_idx"] = np.arange(len(df_stream))

print("Tamanho do stream:", len(df_stream))
print(df_stream.groupby("year")["global_idx"]
      .agg(["min", "max", "count"]).rename(columns={"count": "n_trajs"}))
df_stream.head(3)


In [ ]:
# ============== ADWIN ==============
# delta=0.002 é o default da literatura (Bifet & Gavaldà 2007). Quanto MENOR delta,
# mais conservador (menos falsos positivos, mais latência). Se houver muitos
# alarmes, suba para 0.01; se houver poucos, baixe para 0.0005.
adwin = river_drift.ADWIN(delta=0.002)
adwin_alarms = []
for i, v in enumerate(df_stream["ade_traj"].values):
    adwin.update(float(v))
    if adwin.drift_detected:
        adwin_alarms.append(i)

ALARMS["ADWIN_ADE_seq2seq_30_15"] = adwin_alarms
print(f"ADWIN: {len(adwin_alarms)} alarme(s) em {len(df_stream)} amostras")
print("Primeiros índices:", adwin_alarms[:10])


In [ ]:
# ============== Page-Hinkley ==============
# delta = slack tolerado (qto MAIOR, mais robusto a ruído).
# threshold = magnitude acumulada para disparar. Para ADE em mm com p50~10,
# threshold=50 é um ponto de partida razoável.
# Calibração: se houver MUITO alarme, suba threshold; se houver POUCO, abaixe.
ph = river_drift.PageHinkley(min_instances=30, delta=0.005, threshold=50.0,
                              alpha=1 - 1e-4)
ph_alarms = []
for i, v in enumerate(df_stream["ade_traj"].values):
    ph.update(float(v))
    if ph.drift_detected:
        ph_alarms.append(i)

ALARMS["PageHinkley_ADE_seq2seq_30_15"] = ph_alarms
print(f"Page-Hinkley: {len(ph_alarms)} alarme(s) em {len(df_stream)} amostras")
print("Primeiros índices:", ph_alarms[:10])


In [ ]:
# Plot: série de erro com linhas verticais nos alarmes de cada detector
def make_plot_concept_drift(lang):
    fig, ax = plt.subplots(figsize=(12, 5))

    # série bruta (cinza claro) + média móvel para legibilidade
    x = df_stream["global_idx"].values
    y = df_stream["ade_traj"].values
    ax.plot(x, y, color="lightgray", lw=0.6, alpha=0.7,
            label=T("ADE bruto", "Raw ADE", lang))
    win = max(50, len(df_stream) // 100)
    if len(df_stream) > win:
        roll = pd.Series(y).rolling(win, min_periods=1).mean().values
        ax.plot(x, roll, color="black", lw=1.5,
                label=T(f"Média móvel ({win})", f"Rolling mean ({win})", lang))

    # marcações coloridas das fronteiras de ANO (referência visual)
    for y_ in sorted(df_stream["year"].dropna().unique().astype(int)):
        sub = df_stream[df_stream["year"] == y_]
        if not len(sub):
            continue
        x0 = sub["global_idx"].min()
        ax.axvline(x0, color=YEAR_COLORS.get(y_, "gray"),
                    lw=0.8, alpha=0.4, linestyle=":")
        ax.text(x0, ax.get_ylim()[1] * 0.95 if False else max(y) * 0.95,
                str(y_), color=YEAR_COLORS.get(y_, "gray"),
                fontsize=9, ha="left", va="top", alpha=0.8)

    # alarmes ADWIN (azul)
    for idx in adwin_alarms:
        ax.axvline(idx, color="#2A6FDB", lw=1.2, linestyle="--", alpha=0.75)
    if adwin_alarms:
        ax.axvline(adwin_alarms[0], color="#2A6FDB", lw=1.2, linestyle="--",
                    label=T(f"ADWIN ({len(adwin_alarms)} alarmes)",
                              f"ADWIN ({len(adwin_alarms)} alarms)", lang))

    # alarmes Page-Hinkley (vermelho)
    for idx in ph_alarms:
        ax.axvline(idx, color="#D1495B", lw=1.2, linestyle="-.", alpha=0.75)
    if ph_alarms:
        ax.axvline(ph_alarms[0], color="#D1495B", lw=1.2, linestyle="-.",
                    label=T(f"Page-Hinkley ({len(ph_alarms)} alarmes)",
                              f"Page-Hinkley ({len(ph_alarms)} alarms)", lang))

    ax.set_xlabel(T("Trajetória (ordem temporal global)",
                    "Trajectory (global temporal order)", lang))
    ax.set_ylabel(T("ADE (mm)", "ADE (mm)", lang))
    ax.set_title(T("Detecção online de concept drift — ADE Seq2Seq 30→15",
                    "Online concept-drift detection — Seq2Seq 30→15 ADE", lang))
    ax.legend(loc="upper left", fontsize=9)
    return fig


for lang in LANGS:
    fig = make_plot_concept_drift(lang)
    save_fig_mes2(fig, "6_concept_drift_adwin_ph")
print("-> 6_concept_drift_adwin_ph_*.pdf")


# 7. Covariate shift nas features — KSWIN

Aplicamos o **KSWIN** sobre as séries `ks_speed_window` e `ks_accel_window` de
`df_win`. KSWIN compara, via teste de Kolmogorov-Smirnov, uma janela "antiga"
(`window_size` amostras) com uma janela "recente" (`stat_size` amostras).
Quando p < α, dispara alarme. Aqui estamos monitorando a *distância ao
baseline* de cada janela — se ela passar a ser estatisticamente maior do que
era, KSWIN avisa.


In [ ]:
# Ordena df_win e prepara stream
df_win_sorted = (df_win.sort_values(["year", "match_id", "window_id"])
                 .reset_index(drop=True))
df_win_sorted["global_idx"] = np.arange(len(df_win_sorted))

# IMPORTANTE: parâmetros calibrados para o tamanho do stream (~38 janelas).
# Se você reduzir N_TRAJS_PER_WINDOW no script (e gerar muito mais janelas),
# aumente window_size e stat_size aqui (e.g. 100 / 30 — defaults da lib).
KSWIN_PARAMS = dict(alpha=0.005, window_size=15, stat_size=5, seed=SEED)
print("KSWIN params:", KSWIN_PARAMS)
print("Tamanho do stream KSWIN:", len(df_win_sorted))


In [ ]:
def run_kswin(series_values, params):
    """Roda KSWIN sobre uma série; retorna lista de índices de alarme."""
    det = river_drift.KSWIN(**params)
    alarms = []
    for i, v in enumerate(series_values):
        if not np.isfinite(v):
            continue
        det.update(float(v))
        if det.drift_detected:
            alarms.append(i)
    return alarms


for metric in ["ks_speed_window", "ks_accel_window"]:
    alarms = run_kswin(df_win_sorted[metric].values, KSWIN_PARAMS)
    ALARMS[f"KSWIN_{metric}"] = alarms
    print(f"  KSWIN({metric}): {len(alarms)} alarme(s) -> idx {alarms}")


In [ ]:
def make_plot_kswin_timeline(metric, lang):
    fig, ax = plt.subplots(figsize=(11, 5))
    for y in years_all:
        sub = df_win_sorted[df_win_sorted["year"] == y]
        ax.scatter(sub["global_idx"], sub[metric],
                    color=YEAR_COLORS[y], s=28, alpha=0.85, label=str(y),
                    edgecolor="white", linewidth=0.6)

    alarms = ALARMS.get(f"KSWIN_{metric}", [])
    for idx in alarms:
        ax.axvline(idx, color="black", lw=1.2, linestyle="--", alpha=0.8)
    if alarms:
        ax.axvline(alarms[0], color="black", lw=1.2, linestyle="--",
                    label=T(f"KSWIN ({len(alarms)} alarmes)",
                              f"KSWIN ({len(alarms)} alarms)", lang))

    ax.set_xlabel(T("Janela (ordem temporal global)",
                     "Window (global temporal order)", lang))
    ax.set_ylabel(metric.replace("_window", "").upper().replace("_", " "))
    ax.set_title(T(f"KSWIN sobre {metric}", f"KSWIN over {metric}", lang))
    ax.legend(ncol=2, fontsize=9, loc="best")
    return fig


for metric in ["ks_speed_window", "ks_accel_window"]:
    for lang in LANGS:
        fig = make_plot_kswin_timeline(metric, lang)
        save_fig_mes2(fig, f"7_kswin_timeline_{metric}")
print("-> 7_kswin_timeline_*.pdf")


# 8. Benchmark offline — `ruptures`

`ruptures` faz detecção **retrospectiva** de change-points (vê todo o sinal
antes de decidir). Aqui usamos PELT com custo RBF (robusto a mudanças de
distribuição, não só de média) sobre a série de **ADE médio por jogo** do
Seq2Seq 30→15. Os pontos detectados servem como benchmark "offline" para
comparar com os alarmes online (Seções 6 e 7).


In [ ]:
# Série: ADE médio por jogo, Seq2Seq 30→15, na ordem temporal
ade_per_game = (df_ade_game[
        (df_ade_game["model"] == "Seq2Seq")
        & (df_ade_game["horizon"] == "30→15")
    ]
    .sort_values(["year", "log_file"])
    .reset_index(drop=True)
)
ade_per_game["global_idx"] = np.arange(len(ade_per_game))
print("Tamanho da série (ADE médio por jogo):", len(ade_per_game))
ade_per_game[["log_file", "year", "ade", "match_id"]].head()


In [ ]:
signal = ade_per_game["ade"].astype(float).values

# Pelt + custo RBF: Pelt é exato e eficiente; RBF capta mudanças além da média.
# `pen` é o termo de penalidade — quanto MAIOR, MENOS pontos de quebra.
# Calibração rápida: pen=10 é um ponto de partida; suba para 20-30 se houver
# muitos breakpoints, baixe para 3-5 se o algoritmo não detectar nada.
PEN = 10.0
algo = rpt.Pelt(model="rbf").fit(signal)
breakpoints_pelt = algo.predict(pen=PEN)  # inclui o índice final
# remove o último (que é o tamanho do sinal)
bp_idx_pelt = [b - 1 for b in breakpoints_pelt[:-1]]
ALARMS["Ruptures_Pelt_RBF_ADE_per_game"] = bp_idx_pelt
print(f"Pelt(RBF, pen={PEN}): {len(bp_idx_pelt)} breakpoint(s) em {len(signal)} jogos")
print("Índices:", bp_idx_pelt)


In [ ]:
# Comparação visual: série + breakpoints + alarmes online "projetados"
def make_plot_ruptures(lang):
    fig, ax = plt.subplots(figsize=(12, 5))
    x = ade_per_game["global_idx"].values
    y = ade_per_game["ade"].values

    # cor por ano
    for yr in sorted(ade_per_game["year"].dropna().unique().astype(int)):
        sub = ade_per_game[ade_per_game["year"] == yr]
        ax.scatter(sub["global_idx"], sub["ade"],
                    color=YEAR_COLORS.get(yr, "C0"), s=55, alpha=0.85,
                    edgecolor="white", linewidth=0.6, label=str(yr))

    # linha da série
    ax.plot(x, y, color="black", lw=1.0, alpha=0.5)

    # pintar regimes entre breakpoints (Pelt)
    seg_starts = [0] + [b for b in breakpoints_pelt[:-1]]
    seg_ends   = breakpoints_pelt
    for s, e in zip(seg_starts, seg_ends):
        if e > s:
            mu = float(np.mean(y[s:e]))
            ax.hlines(mu, s, e - 1, color="#2A6FDB", lw=2.5, alpha=0.7)

    for idx in bp_idx_pelt:
        ax.axvline(idx + 0.5, color="#D1495B", lw=1.5, linestyle="--", alpha=0.85)
    if bp_idx_pelt:
        ax.axvline(bp_idx_pelt[0] + 0.5, color="#D1495B", lw=1.5,
                    linestyle="--",
                    label=T(f"Pelt RBF ({len(bp_idx_pelt)} breakpoints)",
                              f"Pelt RBF ({len(bp_idx_pelt)} breakpoints)", lang))

    ax.set_xlabel(T("Jogo (ordem temporal global)",
                     "Game (global temporal order)", lang))
    ax.set_ylabel(T("ADE médio (mm) — Seq2Seq 30→15",
                     "Mean ADE (mm) — Seq2Seq 30→15", lang))
    ax.set_title(T("Benchmark offline: Pelt + RBF sobre ADE por jogo",
                     "Offline benchmark: Pelt + RBF on per-game ADE", lang))
    ax.legend(ncol=3, fontsize=9, loc="best")
    return fig


for lang in LANGS:
    fig = make_plot_ruptures(lang)
    save_fig_mes2(fig, "8_ruptures_pelt_rbf_per_game")
print("-> 8_ruptures_pelt_rbf_per_game_*.pdf")


# 9. Síntese — tabela consolidada de alarmes

Cada detector emite alarmes em **espaços diferentes**:

- ADWIN / Page-Hinkley → índices de **trajetórias** (`df_stream`)
- KSWIN → índices de **janelas** (`df_win_sorted`)
- Pelt → índices de **jogos** (`ade_per_game`)

Aqui mapeamos cada índice de volta para `(year, match_id)` e juntamos tudo
numa tabela única para discussão.


In [ ]:
def map_idx_to_game(method, idx_list, source_df, drift_type):
    """Recebe índices em um stream e devolve linhas (method, year, match_id, ...)."""
    rows = []
    for idx in idx_list:
        if not (0 <= idx < len(source_df)):
            continue
        r = source_df.iloc[idx]
        rows.append({
            "method": method,
            "drift_type": drift_type,
            "stream_index": int(idx),
            "year":  int(r["year"]) if pd.notna(r.get("year")) else None,
            "match_id": r.get("match_id"),
            "log_file": r.get("log_file") if "log_file" in r else None,
        })
    return rows


# fontes de cada detector
SOURCE_DF = {
    "ADWIN_ADE_seq2seq_30_15":        (df_stream,       "Erro (online)"),
    "PageHinkley_ADE_seq2seq_30_15":  (df_stream,       "Erro (online)"),
    "KSWIN_ks_speed_window":           (df_win_sorted,  "Feature (online)"),
    "KSWIN_ks_accel_window":           (df_win_sorted,  "Feature (online)"),
    "Ruptures_Pelt_RBF_ADE_per_game":  (ade_per_game,   "Erro (offline)"),
}

all_rows = []
for method, alarms in ALARMS.items():
    src_df, drift_type = SOURCE_DF.get(method, (None, "?"))
    if src_df is None:
        continue
    all_rows.extend(map_idx_to_game(method, alarms, src_df, drift_type))

df_alarms = pd.DataFrame(all_rows)
if not df_alarms.empty:
    df_alarms = df_alarms.sort_values(["drift_type", "method", "year", "stream_index"])

df_alarms.to_csv(OUT_DIR_MES2 / "alarms_consolidated.csv", index=False)
print(f"-> alarms_consolidated.csv ({len(df_alarms)} alarme(s))")
df_alarms


In [ ]:
# Sumário cruzado: alarmes por (método, ano)
if not df_alarms.empty:
    summary_alarms = (df_alarms
                      .groupby(["method", "drift_type", "year"], dropna=False)
                      .size()
                      .rename("n_alarmes")
                      .reset_index())
else:
    summary_alarms = pd.DataFrame(columns=["method", "drift_type", "year",
                                            "n_alarmes"])

summary_alarms.to_csv(OUT_DIR_MES2 / "alarms_summary_by_year.csv", index=False)
print("-> alarms_summary_by_year.csv")
summary_alarms


## Discussão — alarmes vs eventos conhecidos da liga

A tabela acima dá uma "verdade-detector" para confrontar com mudanças
**reais** do ambiente RoboCup SSL nesse período. Pontos a considerar ao ler
os alarmes:

- **Hiato 2019 → 2021/2022 (pandemia)** — entre a base de treino (2019) e o
  retorno do mundial presencial em 2022, equipes tiveram tempo para repensar
  hardware e estratégia. É a fronteira mais provável para alarmes em todos
  os detectores: divergência se acumula nas distribuições de velocidade e
  aceleração porque o jogo retorna mais agressivo.
- **Mudanças de formato e divisões** — Division B convive com Division A; o
  número efetivo de robôs em campo varia por divisão e por chave. Jogos de
  divisões diferentes na amostra podem aparecer como falso-positivo do
  detector se ele estiver "vendo" um regime quando na verdade é só outro
  conjunto de equipes.
- **Evolução de hardware** — robôs mais rápidos e com chute mais potente em
  2023-2025 deslocam a cauda das distribuições (`speed_p99`, `accel_p99`).
  Esse é o sinal que KSWIN sobre `ks_speed_window` deveria capturar.
- **Mudanças de regra pontuais** — alterações em "ball placement", número
  máximo de robôs, ou time de ataque/defesa não mudam diretamente as
  trajetórias robô-a-robô, mas mudam a *frequência* de cada tipo de
  trajetória. Detectores sobre KS de features (Seção 7) tendem a captar
  isso melhor do que detectores sobre erro do modelo (Seção 6).

**Coincidências esperadas vs falsos positivos**

- Se ADWIN dispara dentro de **2019** ou de qualquer ano isolado, é provável
  ruído do detector — não há mudança de regime real. Aumentar `delta` (ex.
  `0.005`-`0.01`) ou subir `threshold` do Page-Hinkley ajuda.
- Se ADWIN **e** Pelt apontam para a fronteira entre **2021 e 2022**, esse
  é o caso forte de drift de regime e pode ser usado como evidência
  empírica do efeito do hiato pandêmico.
- KSWIN reagindo simultaneamente em `ks_speed` e `ks_accel` reforça a
  hipótese de mudança real nas dinâmicas, e não só no comportamento do
  modelo.

**Próximo passo (Mês 3)**: tomar os pontos acima como hipóteses, decompor
covariate × concept drift por feature, e medir quanto da queda de ADE é
recuperável só com rebalanceamento de treino vs quanto exige re-treino.

**S?ntese num?rica dos artefatos do M?s 2**

A decomposi??o `covariate ? concept` s? ? estim?vel quando h? jogos
compat?veis suficientes entre anos. Em **2025**, o excesso total de ADE do
Seq2Seq 30?15 foi de **1.089 mm** sobre o baseline de
2019, mas `n_games_compat=0`; portanto a fra??o explicada por concept drift
vs covariate shift **n?o ? estim?vel** para esse ano com os dados atuais. No
?nico ano com decomposi??o v?lida, **2023**, o excesso total foi
**1.107 mm**, dos quais **87.1%**
foram atribu?dos a concept drift e **12.9%**
a covariate shift.

Na valida??o contra null, `ADWIN_seq2seq` apresentou **72**
alarmes, com `p = 1.000` contra a distribui??o nula
(`null_mean=72.0`, `null_p95=72.0`). Na curva
de lat?ncia sint?tica, a lat?ncia mediana do ADWIN para shift de **10 mm** foi
**49 amostras** (`n_detected=15/15`).


In [ ]:
# Caminhos Mês 3
RES_MES3 = ROOT / 'Relas' / 'results' / 'mes3'
RES_MES3.mkdir(parents=True, exist_ok=True)


# Mês 3 — Adaptação ao Data Drift

---

# 13. Importance Weighting (LSIF) — Frente 2

Estimamos pesos $w_i = p_{2019}(x_i)/p_y(x_i)$ (LSIF) para cada trajetória de ano $y$.
O ADE-IW mede quanto do excesso de ADE é explicado por covariate shift.

**recovery_pct** = (ADE_y − ADE_IW) / (ADE_y − ADE_2019) × 100  
Se recovery ≈ 100%: todo o excesso é covariate shift.  
Se recovery ≈ 0%: é concept drift puro.

**ESS_ratio** = (Σw)² / (n · Σw²) — deve ser ≥ 0.3 para pesos confiáveis.


In [ ]:
# --- Seção 13: importance weighting ---
iw_path = RES_MES3 / 'iw_decomposition.csv'
if iw_path.exists():
    df_iw = pd.read_csv(iw_path)
    display(df_iw.round(3))
    print()
    for _, row in df_iw.iterrows():
        ess_ok = row['ess_stable']
        rec = row['recovery_pct']
        print(f"  {int(row['year'])}: ESS={row['ess_ratio']:.2f} "
              f"{'ok' if ess_ok else 'INSTAVEL'} | "
              f"ADE_y={row['ade_y']:.2f} ADE_IW={row['ade_iw']:.2f} "
              f"recovery={rec:.1f}%")
else:
    print('[warn] Execute: python drift_analise/compute_importance_weights.py')


In [ ]:
# --- Seção 13: plots IW ---
for fname in ['iw_ade_comparison.pdf', 'iw_recovery.pdf', 'iw_weights_dist.pdf']:
    p = RES_MES3 / fname
    if p.exists():
        print(f'  OK: {p}')
    else:
        print(f'  MISSING: {p}')
        print('  Execute: MPLBACKEND=Agg python drift_analise/plot_iw_results.py')


# 14. Retreino Seletivo nos Breakpoints — Frente 3

Fine-tuning leve da última Dense do Seq2Seq (encoder congelado),
usando dados antes do breakpoint Pelt para treino e dados depois para avaliação.

**Breakpoint:** transição 2019+2021 → 2022-2025 (detectado por Pelt RBF, pen=1).

**Critério de aceite:** recovery > 20% e n_degraded/n_improved < 0.3.

In [ ]:
# --- Seção 14: resultados do retreino seletivo ---
retrain_path = RES_MES3 / 'retrain_results.csv'
if retrain_path.exists():
    df_ret = pd.read_csv(retrain_path)
    display(df_ret.round(3))
    for _, row in df_ret.iterrows():
        ok_rec = row['recovery_pct'] > 20 if pd.notna(row['recovery_pct']) else False
        ok_cat = (row['n_traj_degraded'] / max(row['n_traj_improved'], 1)) < 0.3
        status = 'ACEITE' if (ok_rec and ok_cat) else 'nao atingido'
        print(f"  {row['breakpoint_label']}: {status} "
              f"recovery={row['recovery_pct']:.1f}% "
              f"cat_ratio={row['catastrophic_ratio']:.2f}")
else:
    print('[warn] Execute: python model_analise/retrain_at_breakpoints.py --penalty 1')


# 15. Validação por Divisão — Frente 4

Verifica se o drift temporal sobrevive ao corte por divisão SSL (A/B).

**Nota:** apenas Division A está mapeada nos dados disponíveis;
jogos de 2021 aparecem como 'Unknown' porque a fonte oficial marca participação 'Both'.

In [ ]:
# --- Seção 15: validação por divisão ---
dec_div_path = RES_MES3 / 'by_division_decomposition.csv'
tab_2x2_path = RES_MES3 / 'by_division_2x2_table.csv'
if dec_div_path.exists():
    print('=== Decomposicao por divisao ===')
    display(pd.read_csv(dec_div_path).round(3))
if tab_2x2_path.exists():
    print()
    print('=== Tabela 2x2 (antes/depois 2022) x divisao ===')
    df2x2 = pd.read_csv(tab_2x2_path)
    display(df2x2.round(3))
    for _, row in df2x2.iterrows():
        delta = row.get('delta', float('nan'))
        if abs(delta) > 0.5:
            verdict = 'SIM — drift sobrevive ao corte'
        elif abs(delta) < 0.5:
            verdict = 'NAO — drift desaparece ao fixar divisao'
        else:
            verdict = 'PARCIAL/INCERTO'
        print(f"  Division {row['division']}: delta={delta:+.3f} mm -> {verdict}")
if not dec_div_path.exists():
    print('[warn] Execute: python drift_analise/division_validation.py')


# 16. Sensibilidade Amostral (n=3 vs n=6) — Frente 1

Compara ADE médio por ano (com CI95 bootstrap) ao duplicar o número de jogos amostrados.
Se delta < 1 mm: amostra n=3 é suficiente; resultados do Mês 2 são robustos.

In [ ]:
# --- Seção 16: sensibilidade amostral ---
sens_path = RES_MES3 / 'sample_size_sensitivity.csv'
if sens_path.exists():
    df_sens = pd.read_csv(sens_path)
    pivot = df_sens.pivot_table(
        index=['year', 'model', 'horizon'],
        columns='n_per_year', values='ade_mean'
    ).round(2)
    display(pivot)
    if 3 in pivot.columns and 6 in pivot.columns:
        diff = (pivot[6] - pivot[3]).abs()
        flag = diff[diff > 1.0]
        if flag.empty:
            print('[OK] ADE estavel entre n=3 e n=6 (delta < 1 mm)')
        else:
            print('[AVISO] instabilidade amostral em:')
            print(flag)
else:
    print('[info] Execute apos n=6 run:')
    print('  python model_analise/sample_size_sensitivity.py')


## Conclusão — Mês 3

Respostas às três perguntas centrais:

1. **Quanto do erro adicional pós-2019 é covariate shift?**  
   Ver `iw_decomposition.csv` — coluna `recovery_pct` por ano.

2. **Fine-tuning leve recupera quantos % do excesso?**  
   Ver `retrain_results.csv` — coluna `recovery_pct` e critério de aceite.

3. **Drift entre anos sobrevive ao corte por divisão?**  
   Ver `by_division_2x2_table.csv` — delta ADE antes/depois de 2022 por divisão.
